In [ ]:
# ============================================================
# Mini-Project: Activity Classification — Task 3 Algorithm
# Nur Kursmethoden: Filter (FIR/IIR), FFT, Autokorrelation,
# Welch Periodogramm, Spectrogram
# ============================================================

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt, welch
from DataProcessor import DataProcessor

# =========================
# KONSTANTEN
# =========================
files = ['rawdata/X22/normal_gehen3.pickle', 'rawdata/X22/schnell_Laufen10.pickle', 'rawdata/X22/rennen1.pickle'];
dp = DataProcessor("rawdata/X22/")
BLOCK_DURATION = 5.0        # Sekunden pro Block (Aufgabenblatt §Task3)
ACTIVITY_MAP = {
    0: "Nicht klassifizierbar",
    1: "Ruhen",
    2: "Normal Gehen",
    3: "Schnell Gehen",
    4: "Rennen"
}

# =========================
# SCHRITT 1: BANDPASS-FILTER
# Kursreferenz: §3.3 Filter Design
# =========================
def bandpass_filter(signal, fs, lowcut=0.5, highcut=5.0, order=4):
    """
    IIR Butterworth Bandpass-Filter.
    """
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

# =========================
# SCHRITT 2: TOTAL ACCELERATION (mean removed)
# =========================
def compute_total_acc(ax, ay, az):
    """Gesamtbeschleunigung ohne Gravitations-Komponente (mean entfernt)"""
    acc_total = np.sqrt(ax**2 + ay**2 + az**2)
    acc_total -= np.mean(acc_total)
    return acc_total

# =========================
# SCHRITT 3: CADENCE via AUTOKORRELATION (KORRIGIERT)
# =========================
def estimate_cadence(block_acc, fs):
    """
    Estimates cadence (steps/min) from a 5-second block of accelerometer data.
    """
    # 1. Compute total acceleration (magnitude) and remove gravity offset
    signal = compute_total_acc(block_acc['x'], block_acc['y'], block_acc['z'])
    
    # 2. Biased Autocorrelation
    corr = np.correlate(signal, signal, mode='full')
    corr = corr / len(signal)          # Biased normalization
    
    # 3. Extract the correct part for positive lags.
    #    The full correlation result is symmetric. corr[len(signal)-1] is the peak at zero lag.
    #    To look for the next step, we take the portion *after* the zero lag.
    pos_corr = corr[len(signal):]       # Positive lags only
    
    # 4. Define search range for a step frequency between 0.8 Hz and 3.5 Hz
    min_lag = int(fs / 3.5)      # Fastest step (~3.5 Hz)
    max_lag = int(fs / 0.8)      # Slowest step (~0.8 Hz)
    max_lag = min(max_lag, len(pos_corr))
    
    # 5. Add a safety check to avoid processing empty arrays
    if min_lag >= len(pos_corr) or max_lag <= min_lag:
        return 0.0                # No valid cadence found in this block

    # 6. Find the highest peak in the valid lag range
    search_region = pos_corr[min_lag:max_lag]
    if len(search_region) == 0:
        return 0.0
    
    peak_idx = np.argmax(search_region)
    peak_lag = min_lag + peak_idx
    
    # 7. Convert the lag at the peak to cadence (steps per minute)
    step_period_sec = peak_lag / fs
    cadence = 60.0 / step_period_sec
    
    # 8. Optional: Add a check for a clear, strong peak (good for debugging)
    # peak_val = search_region[peak_idx]
    # mean_val = np.mean(pos_corr)
    # if peak_val / mean_val < 1.5:   # Low SNR, possibly noisy data
    #     return 0.0
    
    return cadencee

# =========================
# SCHRITT 4: DOMINANTE FREQUENZ via WELCH PERIODOGRAMM
# =========================
def get_dominant_frequency(block_acc, fs):
    """
    Dominante Frequenz via Welch Periodogram.
    Gibt (Frequenz, Amplitude) zurück.
    """
    sig = compute_total_acc(block_acc['x'], block_acc['y'], block_acc['z'])
    sig = bandpass_filter(sig, fs, lowcut=0.5, highcut=5.0)
    
    # Welch: Segment = 2s, Overlap = 50%, Hanning-Fenster
    nperseg = min(2 * fs, len(sig))   # falls Block kürzer als 2s
    freqs, Pxx = welch(sig, fs=fs, nperseg=nperseg, noverlap=nperseg//2, window='hann')
    
    # Nur Frequenzband 0.5 – 3.5 Hz betrachten
    mask = (freqs >= 0.5) & (freqs <= 3.5)
    if not np.any(mask):
        return 0.0, 0.0
    
    freqs_mask = freqs[mask]
    Pxx_mask = Pxx[mask]
    idx_max = np.argmax(Pxx_mask)
    return freqs_mask[idx_max], Pxx_mask[idx_max]

# =========================
# SCHRITT 5: AKTIVITÄTSKLASSIFIKATION (mit elif-Ketten, korrigierte Reihenfolge)
# =========================
def classify_activity(cadence, dominant_freq, freq_amplitude):
    """
    Aktivitätsklassifikation über Schwellwert-Logik.
    Reihenfolge: erst laufen, dann schnell gehen, dann normal gehen, dann ruhen.
    """
    # Rennen (höchste Intensität zuerst)
    if dominant_freq >= 2.0 or cadence > 155:
        return 4
    # Schnell Gehen
    elif 1.4 <= dominant_freq < 2.2 and 100 <= cadence <= 160:
        return 3
    # Normal Gehen
    elif 1.0 <= dominant_freq < 1.6 and 60 <= cadence <= 115:
        return 2
    # Ruhen
    elif cadence < 30 and dominant_freq < 0.5:
        return 1
    # Nicht klassifizierbar
    else:
        return 0

# =========================
# HAUPTFUNKTION: ACTIVITY CLASSIFIER
# =========================
def activity_classifier(acceleration_data, fs):
    """
    INPUTS:
        acceleration_data: DataFrame mit Spalten 'x','y','z' (oder 'acc_x','acc_y','acc_z')
        fs: Sampling-Frequenz in Hz
    OUTPUTS:
        cadence: Liste mit Cadence-Werten pro 5-Sek-Block
        activity: Liste mit Aktivitäts-Integern (0-4) pro Block
    """
    # ---------- Input-Handling ----------
    if isinstance(acceleration_data, pd.DataFrame):
        try:
            acc = {
                'x': acceleration_data['x'].values,
                'y': acceleration_data['y'].values,
                'z': acceleration_data['z'].values
            }
        except KeyError:
            try:
                acc = {
                    'x': acceleration_data['acc_x'].values,
                    'y': acceleration_data['acc_y'].values,
                    'z': acceleration_data['acc_z'].values
                }
            except KeyError:
                print("Fehler: DataFrame hat keine Acc-Spalten (x,y,z oder acc_x,acc_y,acc_z)")
                return [], []
    elif isinstance(acceleration_data, np.ndarray):
        acc_data = np.array(acceleration_data)
        if acc_data.shape[1] == 3:
            acc = {'x': acc_data[:,0], 'y': acc_data[:,1], 'z': acc_data[:,2]}
        elif acc_data.shape[0] == 3:
            acc = {'x': acc_data[0,:], 'y': acc_data[1,:], 'z': acc_data[2,:]}
        else:
            print("Fehler: Array-Shape nicht unterstützt")
            return [], []
    else:
        print("Fehler: Input-Typ nicht unterstützt")
        return [], []
    
    N = len(acc['x'])
    block_samples = int(BLOCK_DURATION * fs)
    
    cadence_list = []
    activity_list = []
    
    num_blocks = N // block_samples
    if num_blocks == 0:
        print(f"Warnung: Signallänge ({N} samples) kürzer als Block ({block_samples} samples). Keine Ausgabe.")
        return [], []
    
    for b in range(num_blocks):
        start = b * block_samples
        end = start + block_samples
        
        block_acc = {
            'x': acc['x'][start:end],
            'y': acc['y'][start:end],
            'z': acc['z'][start:end]
        }
        
        cad = estimate_cadence(block_acc, fs)
        cadence_list.append(cad)
        
        dom_freq, freq_amp = get_dominant_frequency(block_acc, fs)
        act = classify_activity(cad, dom_freq, freq_amp)
        activity_list.append(act)
    
    return cadence_list, activity_list


for i in range(3): 
    fileName = files[i]
    dp.loadRawData(fileName)
    for dev in dp.getDevices():
        dp.loadRawDataDevice(dev)
        
cad, act = activity_classifier(dp.dfAcc, dp.fs)
print(f"({fileName}) ---")
print(f"Samplingrate: {dp.fs} Hz")
print(f"Anzahl 5‑Sekunden‑Blöcke: {len(cad)}")
if len(cad) > 0:
    print(f"Cadences (steps/min): {[f'{c:.1f}' for c in cad]}")
    print(f"Aktivitäten: {[ACTIVITY_MAP[a] for a in act]}")
    print(f"Mittlere Cadence: {np.mean(cad):.1f} steps/min")
else:
        print("Keine vollständigen Blöcke.")

"""
# =========================
# TEST MIT DEINEN MESSDATEN (angepasst an existierende Dateien aus Exercise 1)
# =========================
if __name__ == "__main__":
    # Verwende den gleichen DataProcessor-Pfad wie in Exercise 1
    dp = DataProcessor("rawdata/X22/")
    
    # Dateien, die in Exercise 1 erfolgreich geladen wurden:
    test_files = [
        ("rawdata/X22/Laufdata2.pickle",        "Normal Gehen"),
        ("rawdata/X22/schnell_Laufen10.pickle", "Schnell Gehen"),
        ("rawdata/X22/rennen1.pickle",          "Rennen")
    ]
    
    for pfad, name in test_files:
        print(f"\n{'='*60}")
        print(f"Test: {name} — Datei: {pfad}")
        print(f"{'='*60}")
        
        try:
            dp.loadRawData(pfad)
            devices = dp.getDevices()
            if not devices:
                print("Keine Devices gefunden.")
                continue
            for dev in devices:
                dp.loadRawDataDevice(dev)
            
            # Sampling-Frequenz aus dem geladenen Objekt holen
            fs = dp.fs
            print(f"Sampling-Frequenz: {fs} Hz")
            
            cadences, activities = activity_classifier(dp.dfAcc, fs)
            
            print(f"Anzahl Blöcke (à {BLOCK_DURATION} s): {len(cadences)}")
            if len(cadences) > 0:
                print(f"Cadence [Schritte/min]: {[f'{c:.1f}' for c in cadences]}")
                print(f"Aktivität: {[ACTIVITY_MAP[a] for a in activities]}")
                print(f"Mittlere Cadence: {np.mean(cadences):.1f} Schritte/min")
            else:
                print("Keine vollständigen Blöcke vorhanden.")
        except FileNotFoundError:
            print(f"Datei nicht gefunden: {pfad}. Überspringe.")
        except Exception as e:
            print(f"Fehler beim Laden/Verarbeiten von {pfad}: {e}")
"""